<a href="https://colab.research.google.com/github/abhi8823/machine-learning-lab/blob/main/weather_dataset_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from typing import Dict, Any, Optional

class WeatherDataPipeline:
    """A robust pipeline for fetching, analyzing, and exporting city weather forecasts via Open-Meteo."""

    GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
    FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

    def __init__(self, city_name: str):
        self.city_name = city_name.strip()
        self.latitude: Optional[float] = None
        self.longitude: Optional[float] = None
        self.df: Optional[pd.DataFrame] = None

    def fetch_coordinates(self) -> None:
        """Retrieves latitude and longitude for the specified city."""
        params = {
            "name": self.city_name,
            "count": 1,
            "language": "en",
            "format": "json"
        }
        response = requests.get(self.GEOCODE_URL, params=params)
        response.raise_for_status()
        data = response.json()

        if "results" not in data or not data["results"]:
            raise ValueError(f"Could not find geographic coordinates for '{self.city_name}'.")

        top_match = data["results"][0]
        self.latitude = top_match["latitude"]
        self.longitude = top_match["longitude"]
        print(f"✔ Located '{top_match.get('name', self.city_name)}': Lat {self.latitude}, Lon {self.longitude}")

    def fetch_forecast(self) -> None:
        """Fetches hourly weather metrics for the resolved coordinates[cite: 1]."""
        if self.latitude is None or self.longitude is None:
            raise RuntimeError("Coordinates must be fetched before requesting weather data.")

        params = {
            "latitude": self.latitude,
            "longitude": self.longitude,
            "hourly": "temperature_2m,relative_humidity_2m,precipitation_probability,rain,cloud_cover",
            "timezone": "auto"
        }
        response = requests.get(self.FORECAST_URL, params=params)
        response.raise_for_status()
        weather_data = response.json()

        if "hourly" not in weather_data:
            raise KeyError("The weather API response did not contain hourly data metrics.")

        self.df = pd.DataFrame(weather_data["hourly"])
        self.df["time"] = pd.to_datetime(self.df["time"])

    def generate_summary(self) -> None:
        """Computes and prints analytical insights from the fetched dataset."""
        if self.df is None or self.df.empty:
            print("No data available to summarize.")
            return

        print("\n--- Atmospheric Summary Report ---")
        print(f"Total Forecast Hours Tracked : {len(self.df)}")
        print(f"Temperature Range            : {self.df['temperature_2m'].min()}°C to {self.df['temperature_2m'].max()}°C")
        print(f"Average Relative Humidity    : {self.df['relative_humidity_2m'].mean():.1f}%")
        print(f"Peak Rain Probability        : {self.df['precipitation_probability'].max()}%")
        print(f"Total Expected Rainfall      : {self.df['rain'].sum():.2f} mm")
        print("----------------------------------\n")

    def export_to_csv(self) -> str:
        """Exports the processed dataframe to a CSV file[cite: 1]."""
        if self.df is None or self.df.empty:
            raise ValueError("No dataframe exists to export.")

        filename = f"{self.city_name.lower().replace(' ', '_')}_weather.csv"
        self.df.to_csv(filename, index=False)
        print(f"✔ Successfully exported dataset to '{filename}'")
        return filename

    def run(self) -> None:
        """Executes the complete end-to-end data pipeline."""
        print(f"Initializing pipeline for: {self.city_name}")
        self.fetch_coordinates()
        self.fetch_forecast()
        self.generate_summary()
        self.export_to_csv()


# Example Execution
if __name__ == "__main__":
    target_city = "Dewas"
    pipeline = WeatherDataPipeline(target_city)
    pipeline.run()

Initializing pipeline for: Dewas
✔ Located 'Dewas': Lat 22.96585, Lon 76.05526

--- Atmospheric Summary Report ---
Total Forecast Hours Tracked : 168
Temperature Range            : 18.5°C to 31.5°C
Average Relative Humidity    : 45.6%
Peak Rain Probability        : 24%
Total Expected Rainfall      : 0.00 mm
----------------------------------

✔ Successfully exported dataset to 'dewas_weather.csv'
